# Import log files into panda dataframes

In [1]:
import pandas as pd
from glob import glob
import os


# List all .dat files in the current directory
# elephant
dat_file_root = "/srv/data/stratbox_simulations/stratbox_particle_runs/bx5/smd132/sn34/pe300/4pc_resume/4pc"

dat_files = glob(os.path.join(dat_file_root, "SNfeedback.dat"))

# Initialize an empty DataFrame
all_data = pd.DataFrame()

# Read and concatenate data from all .dat files
for dat_file in dat_files:
    # Assuming space-separated values in the .dat files
    df = pd.read_csv(dat_file, delim_whitespace=True, header=None,
                     names=['n_SN', 'type', 'n_timestep', 'n_tracer', 'time',
                            'posx', 'posy', 'posz', 'radius', 'mass'])
    
    # Convert the columns to numerical
    df = df.iloc[1:]
    df['n_SN'] = df['n_SN'].map(int)
    df['type'] = df['type'].map(int)
    df['n_timestep'] = df['n_timestep'].map(int)
    df['n_tracer'] = df['n_tracer'].map(int)
    df['time'] = pd.to_numeric(df['time'],errors='coerce')
    df['posx'] = pd.to_numeric(df['posx'],errors='coerce')
    df['posy'] = pd.to_numeric(df['posy'],errors='coerce')
    df['posz'] = pd.to_numeric(df['posz'],errors='coerce')
    df['radius'] = pd.to_numeric(df['radius'],errors='coerce')
    df['mass'] = pd.to_numeric(df['mass'],errors='coerce')
    all_data = pd.concat([all_data, df], ignore_index=True)
    all_data = all_data.drop(df[df['n_tracer'] != 0].index)

all_data.head()


,n_SN,type,n_timestep,n_tracer,time,posx,posy,posz,radius,mass
0,1,2,12,0,2.869811e+12,9.763277e+20,1.530785e+21,-1.072755e+21,1.243486e+20,4.098446e+35
1,2,1,20,0,4.796885e+12,-1.145076e+21,-1.434358e+21,-1.205343e+19,3.511398e+19,9.042147e+35
2,3,2,26,0,5.738623e+12,3.977631e+20,-1.337930e+21,4.700837e+20,4.278450e+19,4.298436e+35
3,4,2,34,0,8.607434e+12,-1.120969e+21,1.482572e+21,-1.084808e+20,3.511398e+19,8.470546e+35
4,5,1,40,0,9.592771e+12,1.434358e+21,-8.557934e+20,2.205777e+21,3.085678e+20,4.513983e+33


In [2]:
# convert seconds to Megayears
def seconds_to_megayears(seconds):
    return seconds / (1e6 * 365 * 24 * 3600)

# Convert pixel value to pc
def pixel2pc(coord, x_y_z, top_z = 500):
    if x_y_z == "x":
        return coord - top_z
    elif x_y_z == "y":
        return top_z - coord
    elif x_y_z == "z":
        return coord - top_z
    return coord

def pix_256_2pc(pix_256):
    return pix_256 * (1000 / 256)

def pc2pix_256(pc):
    return pc * (256 / 1000)

def cm2pc(cm):
    return cm * 3.24077929e-19

# filter the DataFrame
def filter_data(df, range_coord):
    return df[(df['posx_pc'] > range_coord[0]) & (df['posx_pc'] < range_coord[0] + range_coord[2]) & 
              (df['posy_pc'] > range_coord[1]) & (df['posy_pc'] < range_coord[1] + range_coord[3]) & 
              (df['posz_pc'] > range_coord[4]) & (df['posz_pc'] < range_coord[5])]

def timestamp2Myr(timestamp):
    return (timestamp - 200) * 0.1 + 191

# Convert time to Megayears
all_data['time_Myr'] = seconds_to_megayears(all_data['time'])

# Convert 'pos' from centimeters to parsecs
all_data['posx_pc'] = cm2pc(all_data['posx'])
all_data['posy_pc'] = cm2pc(all_data['posy'])
all_data['posz_pc'] = cm2pc(all_data['posz'])

# Sort the DataFrame by time in ascending order
all_data.sort_values(by='time_Myr', inplace=True)

In [6]:
low_x0, low_y0, low_w, low_h, bottom_z, top_z = -300 , -450, 50, 50, -100, 0
# low_x0, low_y0, low_w, low_h = pixel2pc(low_x0, "x"), pixel2pc(low_y0, "y"), pixel2pc(low_w), pixel2pc(low_h)

In [3]:
all_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 12250 entries, 0 to 14124
Data columns (total 14 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   n_SN        12250 non-null  int64  
 1   type        12250 non-null  int64  
 2   n_timestep  12250 non-null  int64  
 3   n_tracer    12250 non-null  int64  
 4   time        12250 non-null  float64
 5   posx        12250 non-null  float64
 6   posy        12250 non-null  float64
 7   posz        12250 non-null  float64
 8   radius      12250 non-null  float64
 9   mass        12250 non-null  float64
 10  time_Myr    12250 non-null  float64
 11  posx_pc     12250 non-null  float64
 12  posy_pc     12250 non-null  float64
 13  posz_pc     12250 non-null  float64
dtypes: float64(10), int64(4)
memory usage: 1.4 MB


In [6]:
low_x0, low_y0, low_w, low_h, bottom_z, top_z = 0, 0, 500, 500, 0, 500

In [7]:
start_yr = 209
end_yr = start_yr + 18

# Filter data based on specified conditions
# filtered_data = all_data[(all_data['time_Myr'] >= start_yr) & (all_data['time_Myr'] <= end_yr)]
filtered_data = filter_data(all_data[(all_data['time_Myr'] >= start_yr) & (all_data['time_Myr'] <= end_yr)],
                            (low_x0, low_y0, low_w, low_h, bottom_z, top_z))
# filtered_data = filter_data(all_data[(all_data['time_Myr'] >= start_yr) & (all_data['time_Myr'] <= end_yr)], (-92, -101, 50, 50, -400, 400))


# Print the resulting DataFrame
filtered_data
# filtered_data.iloc[0]["posx_pc"]

,n_SN,type,n_timestep,n_tracer,time,posx,posy,posz,radius,mass,time_Myr,posx_pc,posy_pc,posz_pc
6803,6804,3,53852,0,6.592359e+15,2.591487e+20,6.086981e+20,1.627213e+20,9.810099e+19,5.541941e+35,209.042342,83.984374,197.265626,52.734375
6810,6811,1,53930,0,6.599139e+15,6.026714e+18,1.018515e+21,1.386144e+20,2.770208e+19,4.284715e+35,209.257334,1.953125,330.078135,44.921874
6821,6822,2,54044,0,6.609743e+15,5.002173e+20,1.500652e+21,1.988816e+20,3.652930e+19,4.415375e+35,209.593563,162.109374,486.328127,64.453124
6829,6830,1,54140,0,6.618323e+15,1.054675e+21,1.151102e+21,5.845913e+20,1.243486e+20,4.295038e+35,209.865642,341.796890,373.046882,189.453125
6832,6833,2,54172,0,6.621218e+15,1.102889e+21,5.424043e+19,7.834728e+19,1.724158e+19,9.173707e+35,209.957439,357.421886,17.578125,25.390625
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7376,7377,1,60332,0,7.136278e+15,1.024541e+20,7.894995e+20,8.136064e+20,1.020551e+20,3.700619e+35,226.289907,33.203126,255.859376,263.671874
7378,7379,3,60362,0,7.139526e+15,9.944078e+20,1.380118e+21,2.109350e+20,1.846094e+20,7.706008e+35,226.392872,322.265624,447.265621,68.359375
7382,7383,2,60401,0,7.143350e+15,3.676296e+20,7.292324e+20,1.024541e+20,6.607856e+19,4.732383e+35,226.514136,119.140626,236.328126,33.203126
7385,7386,3,60427,0,7.145730e+15,1.042622e+21,1.042622e+21,6.629385e+19,2.740733e+20,9.744261e+35,226.589599,337.890616,337.890616,21.484375


In [ ]:
filtered_data.drop(columns=['posx', 'posy', 'posz', 'time'], inplace=True)
filtered_data.to_csv('SNfeedback_185_200.txt', sep='\t', index=False, encoding='utf-8')

In [18]:
def pc2pix_256(pc):
    return pc * (256 / 1000)

In [20]:
new_posx = pc2pix_256(filtered_data['posx_pc']) + 128
new_posx

6239    42.500000
6254    46.500000
6300    34.500002
6302    43.499998
6345    46.500000
6426    46.500000
6463    34.500002
6516    46.500000
6598    46.500000
6684    46.500000
Name: posx_pc, dtype: float64